# Search Lab — Week 1: Problem Formulation, BFS & DFS
**Course:** Fundamentals of Artificial Intelligence
**Duration:** ~2 hours | **Prerequisites:** Lab 1 & Lab 2 completed

## What You Will Learn
- How to formally define a search problem (without OOP!)
- How to represent states, actions, and goals in Python
- Implement Breadth-First Search (BFS) from scratch
- Implement Depth-First Search (DFS) from scratch
- Compare BFS and DFS on the same problem

## Lab Structure
- **Part 1:** Problem Formulation (30 min)
- **Part 2:** Breadth-First Search (40 min)
- **Part 3:** Depth-First Search (30 min)
- **Part 4:** Exercises (20 min)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualization library  ─  run this cell first
# ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import fai_viz
print("✅ fai_viz visualization library loaded")

## PART 1: Problem Formulation

### What is a Search Problem?
A search problem has **four components**:
1. **Initial State**: Where we start
2. **Actions**: What we can do in each state
3. **Goal Test**: How we know we've succeeded
4. **Path Cost**: The cost of a sequence of actions

> **Key design:** Each action directly returns the resulting state.
> `get_actions(problem, state)` returns `[(action_name, next_state), ...]` — no separate function needed.


### 1.1 Example: Grid Navigation Problem

A robot must navigate from the top-left corner to the bottom-right corner of a grid.

```
Start → [ ][ ][ ][ ][ ]
        [ ][ ][ ][ ][ ]
        [ ][ ][ ][ ][ ]
        [ ][ ][ ][ ][ ]
        [ ][ ][ ][ ][ ] ← Goal
```

**Formulation:**
- **Initial State**: (0, 0)
- **Actions**: `get_actions(problem, state)` → e.g. `[("right", (0,1)), ("down", (1,0)), ...]`
- **Goal Test**: `state == (4, 4)`
- **Path Cost**: 1 per step (unweighted grid)

> Each action already carries its result state — moving right from (0,0) returns `("right", (0,1))`.


### 📌 Problem Reference: 5×5 Grid Navigation

The figure below shows the **grid world** used in this lab. The robot starts at **S = (0, 0)** and must reach **G = (4, 4)**. Right panel: the grid with a wall forcing a detour.

Coordinates are `(row, col)` — row 0 is the **top**, col 0 is the **left**.

In [ ]:
# Show the Grid Navigation reference figure
fai_viz.show_grid_figure()

In [ ]:
# ── Grid Navigation: standalone functions + plain problem dict ───────────
def make_problem(initial, goal, grid_size=5, walls=None):
    """Create a grid navigation search problem."""
    return {
        'initial':   initial,
        'goal':      goal,
        'grid_size': grid_size,
        'walls':     set(walls) if walls else set(),
    }

def is_goal(problem, state):
    """Return True when state matches the goal."""
    return state == problem['goal']

def get_actions(problem, state):
    """Return [(action_name, next_state), ...] for each valid move."""
    row, col = state
    moves = []
    for dr, dc, name in [(-1,0,'UP'),(1,0,'DOWN'),(0,-1,'LEFT'),(0,1,'RIGHT')]:
        nr, nc = row + dr, col + dc
        if 0 <= nr < problem['grid_size'] and 0 <= nc < problem['grid_size']:
            if (nr, nc) not in problem.get('walls', set()):
                moves.append((name, (nr, nc)))
    return moves

def action_cost(problem, state, action):
    """Each step on the grid costs 1."""
    return 1

# Quick test
p = make_problem((0,0), (4,4))
print("Initial:", p['initial'])
print("Goal   :", p['goal'])
print("is_goal((4,4)):", is_goal(p, (4,4)))
print("Actions from (0,0):", get_actions(p, (0,0)))

### 1.2 Nodes in the Search Tree

When we search, we don't just track *states* — we track *nodes*.
A **node** keeps track of:
- `state`: where we are
- `parent`: the node we came from (so we can retrace the path)
- `action`: what action brought us here
- `cost`: total cost from the start (g(n))
- `depth`: how many steps from the start

In [ ]:
def make_node(state, parent=None, action=None, cost=0, depth=0):
    """Create a search tree node (dictionary with state, parent pointer, cost)."""
    return {'state': state, 'parent': parent, 'action': action,
            'cost': cost, 'depth': depth}

def reconstruct_path(node):
    """Walk parent pointers from goal node back to root; return list of states."""
    path = []
    while node is not None:
        path.append(node['state'])
        node = node['parent']
    return list(reversed(path))

def is_cycle(node):
    """Return True if node's state already appears in its ancestor chain.
    Prevents DFS from looping along a single path without a global visited set."""
    ancestor = node['parent']
    while ancestor is not None:
        if ancestor['state'] == node['state']:
            return True
        ancestor = ancestor['parent']
    return False

# Demo
root  = make_node(state=(0,0))
child = make_node(state=(0,1), parent=root, action='RIGHT', cost=1, depth=1)
print('Root:', root['state'], '  Child:', child['state'])
print('Path:', reconstruct_path(child))

### 1.3 The Expand Function

The **expand** function takes a node and a problem, and generates all child nodes by trying every valid action.

In [ ]:
def expand(problem, node, get_actions_fn, action_cost_fn):
    """
    Generate all child nodes from 'node'.
    get_actions_fn(problem, state) → [(action_name, next_state), ...]
    action_cost_fn(problem, state, action) → step cost
    """
    children = []
    for action_name, next_state in get_actions_fn(problem, node['state']):
        cost = action_cost_fn(problem, node['state'], action_name)
        children.append(make_node(
            state  = next_state,
            parent = node,
            action = action_name,
            cost   = node['cost'] + cost,
            depth  = node['depth'] + 1
        ))
    return children

# Demo
root = make_node(state=(0,0))
p1   = make_problem((0,0), (4,4))
ch   = expand(p1, root, get_actions, action_cost)
print(f"Expanding (0,0) -> {len(ch)} children:")
for c in ch:
    print(f"  action={c['action']:6s}  next_state={c['state']}")

### 🎯 Exercise 1 — Formulate the Missionaries & Cannibals Problem

**Problem:** Three missionaries (M) and three cannibals (C) are on the **left bank** of a river. A boat can carry **1 or 2 people** at a time.

**Constraint:** If cannibals ever outnumber missionaries on *either* bank (and there are missionaries present on that bank), the missionaries get eaten. ☠️

**Goal:** Move **everyone** from the left bank to the right bank safely.

---

**Your task — implement two functions:**

**Part A — `mc_is_goal(problem, state)`**
Return `True` when the goal is reached.

**Part B — `mc_get_actions(problem, state)`**
Generate every legal move. Two pieces you must fill in for each candidate move:
1. Compute `new_m` and `new_c` (how the left-bank counts change)
2. Add the move only if it passes the safety check `mc_is_valid(new_m, new_c)`

**State representation:** `(m_left, c_left, boat)`

| Field | Meaning | Range |
|-------|---------|-------|
| `m_left` | missionaries on the **left** bank | 0 – 3 |
| `c_left` | cannibals on the **left** bank | 0 – 3 |
| `boat` | boat position: `0` = left, `1` = right | 0 or 1 |

**Initial state:** `(3, 3, 0)` — everyone on the left, boat on the left

**Goal state:** `(0, 0, 1)` — everyone on the right, boat on the right


**Visualising a move:**
```
State (3, 3, 0) — start           State (1, 2, 1) — after sending 2M+1C
                                  (illegal! 2C > 1M on left)
LEFT ~~~boat~~~> RIGHT             LEFT           RIGHT
 MMM  CCC   🚣                      MC   CC    🚣   MM
(3M, 3C)    (0M, 0C)             (1M, 2C)    (2M, 1C)  ← DANGER

State (2, 2, 1) — after 1M+1C cross   (safe)
LEFT         ~~~boat~~~  RIGHT
 MM  CC                  M C  🚣
(2M, 2C)              (1M, 1C)   ✅
```

**Safety rule (provided as `mc_is_valid`):**
- Left bank:  OK if `m_left == 0`  or  `m_left >= c_left`
- Right bank: OK if `m_right == 0` or  `m_right >= c_right`  (where `m_right = 3 - m_left`, etc.)


**Visualization — Run the cell below to see state diagrams:**

In [ ]:
# State diagram: what does a state (m_left, c_left, boat) look like?
fai_viz.show_mc_state_diagram()

# Safety constraint: legal vs illegal states
fai_viz.show_mc_safety_diagram()

In [ ]:
# ── Water Pouring: standalone functions ─────────────────────────────────
def make_pour_problem(initial, goal, capacities):
    return {'initial': initial, 'goal': goal, 'capacities': capacities}

def pour_is_goal(problem, state):
    """Goal is reached when any jug contains exactly goal litres."""
    return problem['goal'] in state

def pour_get_actions(problem, state):
    """Return [(action_name, next_state), ...] for fill/empty/pour operations."""
    caps, n, results = problem['capacities'], len(state), []
    for i in range(n):
        if state[i] < caps[i]:                          # Fill jug i
            new = list(state); new[i] = caps[i]
            results.append((f'Fill {i}', tuple(new)))
        if state[i] > 0:                                # Empty jug i
            new = list(state); new[i] = 0
            results.append((f'Empty {i}', tuple(new)))
        for j in range(n):                              # Pour i -> j
            if i != j and state[i] > 0 and state[j] < caps[j]:
                amt = min(state[i], caps[j] - state[j])
                new = list(state); new[i] -= amt; new[j] += amt
                results.append((f'Pour {i}->{j}', tuple(new)))
    return results

def pour_action_cost(problem, state, action):
    return 1   # each operation costs 1

# Test
pour_p = make_pour_problem((0,0), 4, (3,5))
print("Initial:", pour_p['initial'])
print("Is (1,4) goal?", pour_is_goal(pour_p, (1,4)))   # True
print("Actions from (0,0):", pour_get_actions(pour_p, (0,0)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  EXERCISE 1: Missionaries & Cannibals — Problem Formulation
# ─────────────────────────────────────────────────────────────────────
# State: (m_left, c_left, boat)
#   m_left = missionaries on the LEFT bank (0–3)
#   c_left = cannibals    on the LEFT bank (0–3)
#   boat   = boat side: 0 = left bank, 1 = right bank

def make_mc_problem():
    return {'initial': (3, 3, 0), 'goal': (0, 0, 1),
            'missionaries': 3, 'cannibals': 3}

# ── Part A: Goal Test ────────────────────────────────────────────────
# The goal is: ALL 3M and 3C have crossed to the RIGHT bank.
# That means the left bank has 0M, 0C, and the boat is on the right.
def mc_is_goal(problem, state):
    # YOUR CODE HERE
    # Hint: unpack state as   m_left, c_left, boat = state
    # Goal: m_left == 0 AND c_left == 0 AND boat == 1
    pass

# ── Part B: Safety Constraint ─────────────────────────────────────────
def mc_is_valid(m_left, c_left):
    """Return True if the state is safe on both banks."""
    m_right = 3 - m_left
    c_right = 3 - c_left
    left_ok  = (m_left  == 0) or (m_left  >= c_left)
    right_ok = (m_right == 0) or (m_right >= c_right)
    return left_ok and right_ok

# ── Part C: Actions ───────────────────────────────────────────────────
def mc_get_actions(problem, state):
    # YOUR CODE HERE
    # Hint: boat on left (boat==0) → people cross to right (+1)
    #       boat on right (boat==1) → people cross to left (-1)
    # For each valid combination (dm, dc) in [(1,0),(2,0),(0,1),(0,2),(1,1)]:
    #   compute new state, check validity, append (action_name, new_state)
    pass

def mc_action_cost(problem, state, action):
    return 1

# Quick test (if implemented)
mc_p = make_mc_problem()
print("Initial:", mc_p['initial'], "  Goal:", mc_p['goal'])

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SOLUTION — Exercise 1: Missionaries & Cannibals          ║
# ╚══════════════════════════════════════════════════════════════╝
def make_mc_problem():
    return {'initial': (3, 3, 0), 'goal': (0, 0, 1),
            'missionaries': 3, 'cannibals': 3}

# ── Part A: Goal Test ────────────────────────────────────────────────
def mc_is_goal(problem, state):    # SOLUTION
    m_left, c_left, boat = state
    return m_left == 0 and c_left == 0 and boat == 1

# ── Part B: Safety Constraint ────────────────────────────────────────
def mc_is_valid(m_left, c_left):    # SOLUTION
    m_right = 3 - m_left
    c_right = 3 - c_left
    left_ok  = (m_left  == 0) or (m_left  >= c_left)
    right_ok = (m_right == 0) or (m_right >= c_right)
    return left_ok and right_ok

# ── Part C: Actions ───────────────────────────────────────────────────
def mc_get_actions(problem, state):    # SOLUTION
    m_left, c_left, boat = state
    # boat=0 (left): people cross LEFT→RIGHT, m_left decreases → direction=-1
    # boat=1 (right): people cross RIGHT→LEFT, m_left increases → direction=+1
    direction = 1 if boat == 1 else -1
    results = []
    for dm, dc in [(1,0),(2,0),(0,1),(0,2),(1,1)]:
        nm = m_left + direction * dm
        nc = c_left + direction * dc
        nb_boat = 1 - boat
        if 0 <= nm <= 3 and 0 <= nc <= 3 and mc_is_valid(nm, nc):
            results.append((f'Move {dm}M{dc}C', (nm, nc, nb_boat)))
    return results

def mc_action_cost(problem, state, action):
    return 1

# Test
mc_p = make_mc_problem()
print("MC actions from start:", mc_get_actions(mc_p, (3,3,0)))
print("mc_is_goal (0,0,1):", mc_is_goal(mc_p, (0,0,1)))

## PART 2: BFS & DFS — General Search in Action

In [ ]:
from collections import deque
# ────────────────────────────────────────────────────────────────────────
# General-Search  (exact pseudocode from course slides)
# ────────────────────────────────────────────────────────────────────────
def general_search(problem, queuing_fn, get_actions_fn, action_cost_fn,
                   is_goal_fn=None):
    """
    Function General-Search(p, QUEUING-FN) — course pseudocode.

      frontier = Make-Queue(Make-Node(Initial-State[p]))
      Loop do
        If frontier is empty then return failure
        node = Remove-Front(frontier)
        If Goal-Test[p] on State(node) succeeds then return node
        frontier = QUEUING-FN(frontier, Expand(node, Actions[p]))
      End

    get_actions_fn, action_cost_fn: standalone problem functions.
    is_goal_fn: optional; defaults to  is_goal(problem, state).
    The QUEUING-FN fully controls search strategy — including cycle prevention.
    """
    if is_goal_fn is None: is_goal_fn = is_goal
    frontier = deque([make_node(problem['initial'])])
    while True:
        if not frontier: return None
        node = frontier.popleft()
        if is_goal_fn(problem, node['state']): return node
        queuing_fn(frontier, expand(problem, node, get_actions_fn, action_cost_fn))  # QUEUING-FN

def enqueue_at_end(frontier, children):
    """BFS: ENQUEUE-AT-END — new nodes go to the BACK of the queue."""
    for child in children: frontier.append(child)

def enqueue_at_front(frontier, children):
    """DFS: ENQUEUE-AT-FRONT — new nodes go to the FRONT (LIFO).
    is_cycle filters out any child whose state already appears on the current path,
    preventing DFS from looping without a global visited set."""
    for child in reversed(children):
        if not is_cycle(child):
            frontier.appendleft(child)

def enqueue_by_cost(frontier, children):
    """UCS: sort frontier by path cost g(n)."""
    frontier.extend(children)
    s = sorted(frontier, key=lambda n: n['cost'])
    frontier.clear(); frontier.extend(s)

print('Search functions ready:')
print('  BFS -> general_search(problem, enqueue_at_end,   get_actions, action_cost)')
print('  DFS -> general_search(problem, enqueue_at_front, get_actions, action_cost)')

### 2.1 BFS — Breadth-First Search

BFS uses `enqueue_at_end` → explores all nodes at depth 1 before depth 2 (level by level).
Guaranteed to find the **shortest path** (fewest steps) in an unweighted graph.

In [ ]:
# Solve the 5x5 grid: (0,0) → (4,4)
p_simple = make_problem(initial=(0,0), goal=(4,4))

# BFS = general_search with enqueue_at_end
bfs_result = general_search(p_simple, enqueue_at_end, get_actions, action_cost)
if bfs_result:
    bfs_path = reconstruct_path(bfs_result)
    print(f"BFS found a path of length {bfs_result['depth']} steps")
    print("Path:", bfs_path)

In [ ]:
# ── Visualise the path on the grid ──────────────────────────

def visualise_grid(problem, path=None, title="Grid Navigation"):
    """Matplotlib grid visualisation — delegates to fai_viz.plot_grid_path."""
    fai_viz.plot_grid_path(problem, path, title=title)

fai_viz.plot_grid_path(p_simple,
                        reconstruct_path(bfs_result) if bfs_result else None,
                        title="BFS Path: (0,0) → (4,4)")

### 2.2 DFS — Depth-First Search

DFS uses `enqueue_at_front` → explores as deep as possible before backtracking (LIFO stack order).
**Not guaranteed** to find the shortest path — it finds *a* path, just not necessarily the optimal one.

**Cycle prevention:** `enqueue_at_front` uses `is_cycle(child)` to skip children whose state already appears on the current path — no global visited set needed.

In [ ]:
# DFS = general_search with enqueue_at_front
p_simple = make_problem(initial=(0,0), goal=(4,4))
print("=== DFS (enqueue_at_front) ===")
dfs_result = general_search(p_simple, enqueue_at_front, get_actions, action_cost)
if dfs_result:
    dfs_path = reconstruct_path(dfs_result)
    print(f"DFS found a path of length {dfs_result['depth']} steps")
    print("Path:", dfs_path)
    fai_viz.plot_grid_path(p_simple, dfs_path,
                           title=f"DFS Path — {dfs_result['depth']} steps")

### 2.3 BFS vs DFS — Side-by-Side Comparison

In [ ]:
# Compare BFS vs DFS on the simple grid
p = make_problem((0,0), (4,4))
bfs_node = general_search(p, enqueue_at_end,   get_actions, action_cost)
dfs_node = general_search(p, enqueue_at_front, get_actions, action_cost)
bfs_path = reconstruct_path(bfs_node)
dfs_path = reconstruct_path(dfs_node)
print(f"{'Algorithm':8s}  {'Steps':6s}  Path")
print("-" * 50)
print(f"{'BFS':8s}  {bfs_node['depth']:6d}  {bfs_path}")
print(f"{'DFS':8s}  {dfs_node['depth']:6d}  {dfs_path}")
print()
print("BFS always finds the SHORTEST path (fewest steps).")
print("DFS may find a longer path but explores differently.")

## PART 3: Exercises

###  Exercise 2 — BFS vs DFS on a Grid with Walls

Use the grid problem with a wall blocking most of row 1:
```
Walls: (1,0), (1,1), (1,2), (1,3)
```
Run **both BFS and DFS** and compare the paths found. Which finds a shorter path? Which explores differently?

In [ ]:
# ─────────────────────────────────────────────────────────
# 🎯 EXERCISE 2: BFS vs DFS on a Grid with Walls
# ─────────────────────────────────────────────────────────
walls_ex2 = {(1,0), (1,1), (1,2), (1,3)}
p_ex2 = make_problem(initial=(0,0), goal=(4,4), walls=walls_ex2)
visualise_grid(p_ex2, title="Maze Layout — row 1 blocked")

# YOUR TURN: run BFS and DFS on p_ex2
# result_bfs = general_search(p_ex2, ???, get_actions, action_cost)
# result_dfs = general_search(p_ex2, ???, get_actions, action_cost)
result_bfs = None   # YOUR CODE HERE
result_dfs = None   # YOUR CODE HERE

if result_bfs:
    visualise_grid(p_ex2, reconstruct_path(result_bfs),
                   title=f"BFS: {result_bfs['depth']} steps")
if result_dfs:
    visualise_grid(p_ex2, reconstruct_path(result_dfs),
                   title=f"DFS: {result_dfs['depth']} steps")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SOLUTION — Exercise 2: BFS and DFS on a Maze             ║
# ╚══════════════════════════════════════════════════════════════╝
walls_ex2 = {(1,0), (1,1), (1,2), (1,3)}
p_ex2 = make_problem(initial=(0,0), goal=(4,4), walls=walls_ex2)

# ── BFS ────────────────────────────────────────────────────────────────
result_bfs = general_search(p_ex2, enqueue_at_end,   get_actions, action_cost)   #  SOLUTION
bfs_path   = reconstruct_path(result_bfs)
visualise_grid(p_ex2, bfs_path, title=f"BFS: {result_bfs['depth']} steps")

# ── DFS ────────────────────────────────────────────────────────────────
result_dfs = general_search(p_ex2, enqueue_at_front, get_actions, action_cost)   #  SOLUTION
dfs_path   = reconstruct_path(result_dfs)
visualise_grid(p_ex2, dfs_path, title=f"DFS: {result_dfs['depth']} steps")

###  Exercise 3 — BFS vs DFS on the Water Pouring Problem

The Water Pouring problem functions are already defined above (`make_pour_problem`, `pour_get_actions`, `pour_action_cost`, `pour_is_goal`).

Your task: solve the 3L/5L → 4L problem using **both BFS and DFS**.
- BFS uses `enqueue_at_end` → finds the shortest solution
- DFS uses `enqueue_at_front` → finds *a* solution (may not be shortest)

Compare: How many steps does each need?

In [ ]:
# ─────────────────────────────────────────────────────────
#  EXERCISE 3: Water Pouring — BFS vs DFS
# ─────────────────────────────────────────────────────────
pour_p = make_pour_problem((0, 0), 4, (3, 5))

# ── BFS ───────────────────────────────────────────────────
# result_bfs = general_search(pour_p, ???, pour_get_actions, pour_action_cost, pour_is_goal)
result_bfs = None   # YOUR CODE HERE

# ── DFS ───────────────────────────────────────────────────
# result_dfs = general_search(pour_p, ???, pour_get_actions, pour_action_cost, pour_is_goal)
result_dfs = None   # YOUR CODE HERE

for name, result in [('BFS', result_bfs), ('DFS', result_dfs)]:
    if result:
        path = reconstruct_path(result)
        print(f"{name}: solved in {result['depth']} steps → {path}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SOLUTION — Exercise 3: Water Pouring — BFS vs DFS        ║
# ╚══════════════════════════════════════════════════════════════╝
pour_p = make_pour_problem((0, 0), 4, (3, 5))

# ── BFS ────────────────────────────────────────────────────────────────
result_bfs = general_search(pour_p, enqueue_at_end,   # SOLUTION
                             pour_get_actions, pour_action_cost, pour_is_goal)
# ── DFS ────────────────────────────────────────────────────────────────
result_dfs = general_search(pour_p, enqueue_at_front,  # SOLUTION
                             pour_get_actions, pour_action_cost, pour_is_goal)

for name, result in [('BFS', result_bfs), ('DFS', result_dfs)]:
    if result:
        path = reconstruct_path(result)
        print(f"{name}: {result['depth']} steps → {path}")
        fai_viz.plot_jug_solution(path, pour_p['capacities'],
                                  title=f"{name} Water Pouring (3L,5L) → Goal 4L",
                                  is_goal_fn=lambda s: pour_p['goal'] in s)

###  Exercise 4 — Compare BFS and DFS: Count Nodes Expanded

`bfs_with_count` is already implemented below as a model.

**Your task:** implement `dfs_with_count` (same structure, but use a **stack** instead of a queue).

**Hint — the only differences from BFS:**
| | BFS | DFS |
|--|-----|-----|
| Data structure | `deque` (queue) | `list` (stack) |
| Add to front/back | `frontier.append(child)` | `frontier.append(child)` ✓ same |
| Remove from | `frontier.popleft()` | `frontier.pop()` ← this changes! |

In [ ]:
# ─────────────────────────────────────────────────────────
# 🎯 EXERCISE 4: Compare BFS vs DFS — Count Nodes Expanded
# ─────────────────────────────────────────────────────────
def search_with_count(problem, queuing_fn, get_actions_fn, action_cost_fn, is_goal_fn=None):
    """Run general_search and return (result_node, nodes_expanded).
    Cycle prevention is handled by the queuing_fn (e.g. enqueue_at_front)."""
    if is_goal_fn is None: is_goal_fn = is_goal
    frontier = deque([make_node(problem['initial'])])
    count    = 0
    while True:
        if not frontier:
            return None, count
        node = frontier.popleft()
        count += 1
        if is_goal_fn(problem, node['state']):
            return node, count
        queuing_fn(frontier, expand(problem, node, get_actions_fn, action_cost_fn))

p = make_problem((0,0), (4,4))
bfs_node, bfs_count = search_with_count(p, enqueue_at_end,   get_actions, action_cost)
dfs_node, dfs_count = search_with_count(p, enqueue_at_front, get_actions, action_cost)

print(f"{'Algorithm':8s}  {'Nodes Expanded':15s}  {'Path Length':12s}")
print("-" * 40)
print(f"{'BFS':8s}  {bfs_count:15d}  {bfs_node['depth']:12d}")
print(f"{'DFS':8s}  {dfs_count:15d}  {dfs_node['depth']:12d}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SOLUTION — Exercise 4: Compare BFS vs DFS                ║
# ╚══════════════════════════════════════════════════════════════╝
p = make_problem((0,0), (4,4))
bfs_node, bfs_count = search_with_count(p, enqueue_at_end,   get_actions, action_cost)   # 
dfs_node, dfs_count = search_with_count(p, enqueue_at_front, get_actions, action_cost)   # 

print(f"{'Algorithm':8s}  {'Nodes Expanded':15s}  {'Path Length':12s}")
print("-" * 40)
print(f"{'BFS':8s}  {bfs_count:15d}  {bfs_node['depth']:12d}")
print(f"{'DFS':8s}  {dfs_count:15d}  {dfs_node['depth']:12d}")
print()
print("BFS is level-by-level → expands more nodes near start (all breadth) but finds optimal path.")
print("DFS dives deep first → fewer nodes near start but may find a longer path.")

### 🎯 Exercise 5 — Reflection Questions

Answer in the cell below (just edit this markdown cell):

1. **BFS vs DFS — Completeness:** If a solution exists, will BFS always find it? What about DFS on an infinite graph?

2. **BFS Optimality:** BFS finds the *shortest* path (fewest steps). Is that always the same as the *cheapest* path? When might they differ?

3. **Water Pouring:** Why can you represent the water-pouring problem as a search problem? What is the state? What are the actions?

4. **Visited Set:** What would happen if you removed the `visited` set from BFS on the Romania map (a cyclic graph)? Try it mentally.

5. **DFS Space:** DFS uses O(depth) memory; BFS uses O(branching factor ^ depth) memory. For a very deep tree, which would you prefer?

---
**Model answers below (read after attempting!)**

1. BFS is complete on finite graphs. DFS can loop forever on infinite/cyclic graphs without a visited set.
2. BFS finds fewest-hop path. If edge costs differ (e.g., Romania roads), fewest hops ≠ cheapest cost — UCS is needed.
3. State = (litres in each jug). Actions = Fill/Empty/Pour. Goal = target amount appears in any jug.
4. Without visited, BFS loops forever on cyclic graphs (revisits states endlessly, never terminates).
5. DFS for deep trees — O(depth) << O(b^depth) for large branching factor b.

## Summary

### What You Learned Today

| Concept | Key Points |
|---------|------------|
| Problem Formulation | State, actions, goal test, path cost — all as Python functions |
| Node | Dict with state, parent, action, cost, depth |
| Expand | Generates all children of a node |
| BFS | FIFO queue (deque), finds shortest path, explores level by level |
| DFS | LIFO stack (list), may not find shortest path, explores deep first |

### Next Lab (Week 2)
- **Uniform Cost Search (UCS)**: handles unequal step costs with a priority queue
- **Greedy Best-First Search**: uses a heuristic to guide search toward the goal
- **Romania map problem**: a real-world weighted graph example

## Visual: BFS vs DFS — Exploration Heatmap

Numbers show the order each cell was expanded (0 = first). BFS sweeps level-by-level; DFS plunges deep before backtracking. Which reaches the goal faster on this grid?

In [ ]:

p_ord = make_problem((0,0),(4,4), grid_size=5)
fai_viz.plot_bfs_dfs_exploration(
    p_ord,
    lambda state: get_actions(p_ord, state)
)


## Visual: Water Jug Solution Path

Each pair of bars represents the jug state at one step. Watch the water level in each jug change as BFS finds the optimal pouring sequence.

In [ ]:
pour_p2 = make_pour_problem((0,0), 4, (3,5))
jug_path_node = general_search(pour_p2, enqueue_at_end,
                                pour_get_actions, pour_action_cost, pour_is_goal)
if jug_path_node:
    jug_path = reconstruct_path(jug_path_node)
    fai_viz.plot_jug_solution(jug_path, pour_p2['capacities'],
                              f"Water Jug {pour_p2['capacities']} → Goal: {pour_p2['goal']}L")
    print(f"Solution in {jug_path_node['depth']} steps: {jug_path}")